In [142]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.experimental import enable_iterative_imputer

data = pd.read_csv("../../preliminar_phase_submision/training_data/SyntheticData_Training.csv")
dictionary = pd.read_csv("tmp_datasets/DataDictionary.csv")
target = "inhospital_mortality"

social_features = dictionary[dictionary['Category'].str.contains('Social')]['Feature']
df = data[social_features.tolist()+[target]]


In [143]:
len(dictionary[dictionary.Category.str.contains('Clinical')]['Feature'].tolist())

98

In [144]:
dictionary.Category.value_counts()

Category
Categorical\n(Clinical)           69
Categorical\n(Social)             25
Categorical \n(Clinical)          17
Continuous\n(Clinical)            10
Continuous\n(Social)               5
Continuous\n(Laboratory)           4
Categorical\n(Laboratory)          3
Continuous\n(Recording data)       2
Continuous \n(Clinical)            1
Categorical\n(Patient details)     1
Continuous \n(Patient details)     1
Discrete\n(Clinical)               1
Discrete\n(Patient details)        1
Variable Type                      1
Name: count, dtype: int64

In [145]:
len(social_features), len(dictionary)

(30, 141)

In [146]:
for f in df.columns:
    print(df[f].value_counts())
    print("missing values:", df[f].isna().sum())
    
    print("\n-------------------------------\n")

travelmethod_adm
Motorcycle           1057
Taxi/special hire     940
Private vehicle        64
Walking                52
Ambulance              30
Other                   4
Name: count, dtype: int64
missing values: 1

-------------------------------

traveldist_adm
30 minutes - 1 hour    700
1 - 2 hours            620
< 30 minutes           395
2 - 3 hours            307
3 - 4 hours             97
4 - 8 hours             26
> 8 hours                2
Name: count, dtype: int64
missing values: 1

-------------------------------

badhealthduration_adm
In good health prior to this illness    1811
1 week - 1 month                         140
1 month - 1 year                         139
< 1 week                                  29
> 1 year                                  24
Unknown                                    1
Name: count, dtype: int64
missing values: 4

-------------------------------

caregiver_adm_new
Mother         1911
Grandparent     170
Other            34
Father           31

<u>missing values where "other", "unknown", etc are options:</u> ✅

'travelmethod_adm', 'badhealthduration_adm', 'caregiver_adm_new', 'momhiv_adm', 'lightfuel_adm' ✅

<u>missing values which we'll assume mean no or never</u>: ✅

momagefirstpregknown_adm -> No ✅
waterpure_adm -> No ✅
bednet_adm -> Never ✅
deadchildren_adm ->  0 ✅
alivechildren_adm -> 1 ✅

<u>missing values use most frequent</u>

watersource_adm
cookloc_adm
householdsize_adm


<u>derive from other missing values<u>:

momageknown_adm << is there a value in "momage_adm" then Yes or else No. Should be no unknown values. ✅

traveldist_adm -> get from travelmethod_adm ✅
momalive_adm -> check against caregiver_adm_new ✅

momage_adm >>>> derive from alivechildren_adm + deadchildren_adm ✅

momagefirstpreg_adm << momage_adm, alivechildren_adm + deadchildren_adm ✅

caregiverage_adm -> check against caregiver_adm_new and use mom's age if it exists; if not, then use the mode of caregiverage_adm✅

In [148]:
%%timeit
import pandas as pd

class CustomImputer:
    def __init__(self):
        # store specific modes
        self.mode_values = {}
        self.default_values = {}
        self.no_default_features = ['traveldist_adm',
            'caregiverage_adm',
            'caregivermarried_adm',
            'momalive_adm',
            'momageknown_adm',
            'momedu_adm',
            'watersource_adm',
            'cookfuel_adm___1',
            'cookfuel_adm___2',
            'cookfuel_adm___3',
            'cookfuel_adm___4',
            'cookfuel_adm___5',
            'cookfuel_adm___6',
            'cookfuel_adm___7',
            'cookfuel_adm___8',
            'cookloc_adm',
            'tobacco_adm',
            'momage_adm',
            'momagefirstpreg_adm',
            'householdsize_adm']

        self.mode_traveldist = None  
        self.mode_momage_per_children_group = None
        self.mode_momage_firstpreg_per_children_group = None

    def fit(self, df):
        """
        """
        for col in self.no_default_features:
            self.mode_values[col] = df[col].mode()[0]

        self.default_values = {
            'travelmethod_adm': 'Other',
            'badhealthduration_adm': 'Unknown',
            'caregiver_adm_new': 'Other',
            'momhiv_adm': 'Unknown',
            'lightfuel_adm': 'Other',
            'momagefirstpregknown_adm': 'No',
            'waterpure_adm': 'No',
            'bednet_adm': 'Never',
            'deadchildren_adm': 0.0,
            'alivechildren_adm': 1.0}

        self.mode_traveldist = df.groupby('travelmethod_adm')['traveldist_adm'].agg(
            lambda x: x.mode()[0] if not x.mode().empty else None)
        
        df['number_of_children'] = df['alivechildren_adm'] + df['deadchildren_adm']
        self.mode_momage_per_children_group = df.groupby('number_of_children')['momage_adm'].apply(
            lambda x: x.mode()[0] if not x.mode().empty else None)
        self.mode_momage_firstpreg_per_children_group = df.groupby(['momage_adm', 'number_of_children'])['momagefirstpreg_adm'].apply(
            lambda x: x.mode()[0] if not x.mode().empty else None)

        self.mode_momage_firstpreg_per_children_group = self.mode_momage_firstpreg_per_children_group.fillna(method='ffill').fillna(method='bfill')


    def transform(self, df):
        """
        Fills missing values
        """
        df = df.copy()
        # missing values where "other", "unknown" are options
        df.fillna(self.default_values, inplace=True)

        # missing values based on other features
        df['momageknown_adm'] = np.where(df['momage_adm'].notna(), "Yes", "No")
        df['traveldist_adm'] = df['traveldist_adm'].fillna(df['travelmethod_adm'].map(self.mode_traveldist))

        df.loc[df['momalive_adm'].isna(), 'momalive_adm'] = df.loc[
            df['momalive_adm'].isna(), 'caregiver_adm_new'].apply(
                lambda x: 'Yes' if x == 'Mother' else 'No')
        
        df['number_of_children'] = df['alivechildren_adm'] + df['deadchildren_adm']
        df['momage_adm'] = df['momage_adm'].fillna(df['number_of_children'].map(self.mode_momage_per_children_group))
        
        df['momagefirstpreg_adm'] = df.apply(
            lambda row: self.mode_momage_firstpreg_per_children_group.loc[
            (row['momage_adm'], row['number_of_children'])] 
            if pd.isna(row['momagefirstpreg_adm']) else row['momagefirstpreg_adm'], axis=1)
         
        df['caregiverage_adm'] = df['momage_adm'].where(
            df['caregiver_adm_new'] == 'Mother',
            self.mode_values["caregiverage_adm"])
        
        df.drop(columns=['number_of_children'], inplace=True)
        

        # catch-all to fill all missing values based on mode
        for col, mode_value in self.mode_values.items():
            df[col].fillna(mode_value, inplace=True)

        return df

    def fit_transform(self, df):
        """
        Combines fit and transform methods.
        """
        self.fit(df)
        return self.transform(df)
    
inputer = CustomImputer()
inputer.fit(df)
transformed_df = inputer.transform(df)
transformed_df[f].isna().sum().sum()

95.3 ms ± 3.64 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [149]:
transformed_df.to_csv("tmp_datasets/social_cleaned.csv", index=False)